In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. 終極自動尋路系統 (地毯式搜索)
# ==========================================
DATA_DIR = None

# 在 Kaggle 環境中自動搜尋包含 train.csv 的資料夾
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root
            break

# 如果在本地端 (Colab/本機)，則檢查當前目錄
if DATA_DIR is None and os.path.exists('./train.csv'):
    DATA_DIR = './'

if DATA_DIR is None:
    raise FileNotFoundError("❌ 還是找不到 train.csv！請檢查檔案名稱是否真的是 train.csv")

print(f"✅ 成功找到資料集，讀取路徑: {DATA_DIR}")
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

# ==========================================
# 2. 特徵工程 (Feature Engineering)
# ==========================================
def process_features(df):
    df = df.copy()
    
    # 輪胎退化特徵
    df['Deg_per_Lap'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)
    df['Deg_Acceleration'] = df['Cumulative_Degradation'] * df['TyreLife']
    
    # 移除無泛化能力的合成特徵與 ID
    cols_to_drop = ['id', 'Driver']
    df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)
    
    return df

train_df = process_features(train)
test_df = process_features(test)

X = train_df.drop(columns=['PitNextLap'])
y = train_df['PitNextLap']

# 類別特徵處理：為 XGBoost 準備 Label Encoding
cat_cols = ['Compound', 'Race']
X_xgb = X.copy()
test_xgb = test_df.copy()

for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], test_df[col]]).astype(str))
    
    X_xgb[col] = le.transform(X[col].astype(str))
    test_xgb[col] = le.transform(test_df[col].astype(str))
    
    X[col] = X[col].astype('category')
    test_df[col] = test_df[col].astype('category')

# ==========================================
# 3. K-Fold 交叉驗證訓練
# ==========================================
N_SPLITS = 5
folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
preds_lgb = np.zeros(len(test_df))
preds_xgb = np.zeros(len(test_df))

print(f"開始訓練 {N_SPLITS}-Fold LightGBM 與 XGBoost...\n")

for fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    X_trn_lgb, y_trn = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val_lgb, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    X_trn_xgb, X_val_xgb = X_xgb.iloc[trn_idx], X_xgb.iloc[val_idx]
    
    # --- LightGBM ---
    model_lgb = lgb.LGBMClassifier(
        objective='binary', metric='auc', learning_rate=0.03,
        max_depth=6, num_leaves=31, n_estimators=1500,
        random_state=42+fold, verbose=-1
    )
    model_lgb.fit(
        X_trn_lgb, y_trn,
        eval_set=[(X_val_lgb, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val_lgb)[:, 1]
    preds_lgb += model_lgb.predict_proba(test_df)[:, 1] / N_SPLITS
    
    # --- XGBoost ---
    model_xgb = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='auc', learning_rate=0.03,
        max_depth=6, n_estimators=1500, random_state=42+fold,
        enable_categorical=False
    )
    model_xgb.fit(
        X_trn_xgb, y_trn,
        eval_set=[(X_val_xgb, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val_xgb)[:, 1]
    preds_xgb += model_xgb.predict_proba(test_xgb)[:, 1] / N_SPLITS
    
    lgb_auc = roc_auc_score(y_val, oof_lgb[val_idx])
    xgb_auc = roc_auc_score(y_val, oof_xgb[val_idx])
    print(f"Fold {fold+1} | LGBM AUC: {lgb_auc:.5f} | XGB AUC: {xgb_auc:.5f}")

# ==========================================
# 4. 融合與輸出
# ==========================================
print("\n" + "="*40)
print(f"總結 LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}")
print(f"總結 XGBoost OOF AUC  : {roc_auc_score(y, oof_xgb):.5f}")

final_oof = 0.6 * oof_lgb + 0.4 * oof_xgb
final_preds = 0.6 * preds_lgb + 0.4 * preds_xgb

print(f"🔥 融合後 OOF AUC     : {roc_auc_score(y, final_oof):.5f}")
print("="*40)

sample_sub['PitNextLap'] = final_preds
sample_sub.to_csv('submission.csv', index=False)
print("\n✅ 訓練完成！已成功生成 submission.csv，準備好提交。")

✅ 成功找到資料集，讀取路徑: /kaggle/input/competitions/playground-series-s6e5
開始訓練 5-Fold LightGBM 與 XGBoost...

Fold 1 | LGBM AUC: 0.94861 | XGB AUC: 0.94830
Fold 2 | LGBM AUC: 0.94668 | XGB AUC: 0.94633
Fold 3 | LGBM AUC: 0.94774 | XGB AUC: 0.94731
Fold 4 | LGBM AUC: 0.94690 | XGB AUC: 0.94628
Fold 5 | LGBM AUC: 0.94782 | XGB AUC: 0.94711

總結 LightGBM OOF AUC : 0.94755
總結 XGBoost OOF AUC  : 0.94706
🔥 融合後 OOF AUC     : 0.94802

✅ 訓練完成！已成功生成 submission.csv，準備好提交。
